# 07. AI Analysis: Representation Similarity (CKA, RSA) and TwoNN Intrinsic Dimension

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Kriegeskorte et al. (2008 *Representational similarity analysis: RSA*); Facco et al. (2017 *Estimating the intrinsic dimension of datasets: TwoNN*); Raghu et al. (2017 *SVCCA*); Kornblith et al. (2019 *Similarity of Neural Network Representations Revisited: CKA*).
- **Purpose:** Compare latent geometries across neural networks using Linear Centered Kernel Alignment (CKA), Representational Similarity Analysis (RSA), and estimate manifold compression via TwoNN intrinsic dimension.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [06. AI Brain Surgery Knowledge Editing ROME MEMIT](06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.ipynb) $\to$ `🔗 Downstream Syntheses:` [08. Neural Collapse Superposition and WeightWatcher RMT](08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not evaluate CKA on uncentered Gram matrices (shared mean activation offsets dominate raw inner products, producing spuriously high similarity); avoid TwoNN on small datasets ($N < 50$) where boundary effects distort nearest-neighbor distance ratios.

## Mathematical Foundations & Governing Equations

1. **Double Centering**: $K_c = H K H = K - \frac{1}{N}\mathbf{1}K - \frac{1}{N}K\mathbf{1} + \frac{1}{N^2}\mathbf{1}K\mathbf{1}$
2. **Linear CKA**: $\text{CKA}(X, Y) = \frac{\text{HSIC}(K, L)}{\sqrt{\text{HSIC}(K, K) \cdot \text{HSIC}(L, L)}}, \quad K = X X^T, \ L = Y Y^T$
3. **Representational Similarity Analysis**: $\text{RSA}(A, B) = \text{corr}(\text{triu}(\text{RDM}_A), \text{triu}(\text{RDM}_B))$
4. **TwoNN Intrinsic Dimension**: $\hat{d} = \frac{N}{\sum_{i=1}^N \ln(r_2(i) / r_1(i))}$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_center_gram(gram):
    return gram - gram.mean(dim=0, keepdim=True) - gram.mean(dim=1, keepdim=True) + gram.mean()

def calc_linear_cka(x, y):
    k_c = calc_center_gram(x @ x.t())
    l_c = calc_center_gram(y @ y.t())
    hsic_xy = (k_c * l_c).sum()
    hsic_xx = (k_c * k_c).sum()
    hsic_yy = (l_c * l_c).sum()
    return hsic_xy / (torch.sqrt(hsic_xx * hsic_yy) + 1e-8)

def calc_rsa(rdm_a, rdm_b):
    n = rdm_a.shape[0]
    i, j = torch.triu_indices(n, n, offset=1)
    va = rdm_a[i, j] - rdm_a[i, j].mean()
    vb = rdm_b[i, j] - rdm_b[i, j].mean()
    return (va * vb).sum() / (torch.sqrt((va * va).sum() * (vb * vb).sum()) + 1e-8)

def calc_twonn(r1, r2):
    return float(r1.shape[0]) / (torch.log(r2 / (r1 + 1e-8)).sum() + 1e-8)

In [ ]:
# Parity verification against golden reference tensors
cka = calc_linear_cka(weights["cka_x"], weights["cka_y"])
assert torch.max(torch.abs(cka - weights["cka_score_ref"])) < 1e-5, "CKA mismatch"

rsa = calc_rsa(weights["rdm_a"], weights["rdm_b"])
assert torch.max(torch.abs(rsa - weights["rsa_corr_ref"])) < 1e-5, "RSA correlation mismatch"

twonn_id = calc_twonn(weights["twonn_r1"], weights["twonn_r2"])
assert torch.max(torch.abs(twonn_id - weights["twonn_id_ref"])) < 1e-5, "TwoNN mismatch"
print("All CKA, RSA, and TwoNN assertions passed!")

In [ ]:
# Benchmarking representation geometry operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_linear_cka(weights["cka_x"], weights["cka_y"])
    _ = calc_rsa(weights["rdm_a"], weights["rdm_b"])
    _ = calc_twonn(weights["twonn_r1"], weights["twonn_r2"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused CKA + RSA + TwoNN latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Uncentered Gram Artifact in Representation Alignment
# Two orthogonal representations with a large common mean shift
x_raw = torch.tensor([[1.0, 0.0], [-1.0, 0.0], [1.0, 0.0], [-1.0, 0.0]])
y_raw = torch.tensor([[0.0, 1.0], [0.0, 1.0], [0.0, -1.0], [0.0, -1.0]])
shift = 100.0 * torch.ones(4, 2)
x_shifted = x_raw + shift
y_shifted = y_raw + shift
# True Centered CKA correctly identifies zero alignment
cka_centered = calc_linear_cka(x_shifted, y_shifted).item()
# Naive uncentered inner product is corrupted by the mean shift
k_uncentered = (x_shifted @ x_shifted.t())
l_uncentered = (y_shifted @ y_shifted.t())
raw_cos = ((k_uncentered * l_uncentered).sum() / (torch.norm(k_uncentered) * torch.norm(l_uncentered))).item()
print(f"Centered CKA: {cka_centered:.4f} vs Uncentered Raw Cosine: {raw_cos:.4f}")
assert cka_centered < 0.01 and raw_cos > 0.99, "Centering is mandatory to remove spurious mean shift artifacts!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Linear CKA** provides an orthogonal-rotation-invariant metric for comparing neural representations across architectures.
2. **Double-Centering** is mathematically required to eliminate spurious mean-shift alignment artifacts.
3. **TwoNN** estimates intrinsic manifold dimensionality from the Pareto distribution of nearest-neighbor distance ratios.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.md](../../markdown/10_xai_and_ai_analysis_methods/07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.md)
- **PyTorch 2.x Module:** [07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.py](../../pytorch2/10_xai_and_ai_analysis_methods/07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.py)
- **Functional JAX Module:** [07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.py](../../jax/10_xai_and_ai_analysis_methods/07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.py)
- **Native Rust Candle Track:** [07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.rs](../../candle/10_xai_and_ai_analysis_methods/07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.rs)
- **Hardware Kernel (CPU / WGSL):** [07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.rs](../../kernels/10_xai_and_ai_analysis_methods/07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.rs)
- **Golden SafeTensors:** [07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.safetensors](../../weights/10_xai_and_ai_analysis_methods/07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. AI Brain Surgery Knowledge Editing ROME MEMIT](06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 08. Neural Collapse Superposition and WeightWatcher RMT](08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.ipynb) &rarr;